# PRATYAKSHA · Step 2 — R0 (no lookup), R1 (lookup), R2 (lookup + reorder)

**Goal of this notebook:** find out what *retrieval* and *reranking* buy a small AI model on our
bank-reconciliation task, compared with the no-AI baseline (80.8% clean / 59.9% messy).

Run the cells **top to bottom**. Each cell says **what** it does and **why**.
Before starting: *Runtime → Change runtime type → T4 GPU*.

## 1. Connect Google Drive
**Why:** the project and all results live in Drive, so nothing is lost when Colab resets.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Unpack the Step 2 code over the Step 1 folder
**What:** upload `pratyaksha_step2.zip` into your **Colab Notebooks** folder first, then run this.
**Why:** it adds the new code (retrieval, reranker, AI client) without touching your data or Step 1 results.

In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step2.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!ls pratyaksha

/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Colab Notebooks/pratyaksha
harness  __init__.py  llm  __pycache__	retrieval  systems  world


## 3. Install libraries
**Why:** `transformers` runs the Qwen model, `sentence-transformers` runs the reranker,
`rank-bm25` is the keyword search. Takes about 1–2 minutes.

In [3]:
!pip install -q -U transformers accelerate sentence-transformers rank-bm25 pytest
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

transformers 5.24.0 | torch 2.14.1+cu130 | GPU: Tesla T4


## 4. Rebuild the data and run all tests
**Why:** the data folder is the same as Step 1 (same seed → same data). The tests (22) check the
answer key *and* the new retrieval/prompt code before we spend GPU time.
**Expect:** `22 passed`.

In [4]:
!python -m pratyaksha.world.generate --out data/wwi_in
!python -m pytest -q

{"customers": 60, "suppliers": 8, "invoices": 1361, "bank_entries": 997, "remittances": 501}
......................                                                   [100%]
22 passed in 3.18s


## 5. Smoke test: can the model run at all?
**What:** loads **Qwen3.5-0.8B** (about 1.6 GB download) and asks one tiny question.
**Why:** catches setup problems in 1 minute instead of in the middle of a long run.
**Expect:** `PASS`. If it says `CHECK`, run the next line with `--dtype float32` instead.

In [5]:
!python scripts/smoke_llm.py --model Qwen/Qwen3.5-0.8B

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.65MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 24.5MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 10.5MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 61.3MB/s]

tokenizer.json: downloading bytes:   0% 0.00/12.8M [00:00<?, ?B/s]
tokenizer.json: downloading bytes:  15% 1.91M/12.8M [00:00<00:03, 3.12MB/s, 18.3kB/s  ]
tokenizer.json: downloading bytes: 100% 5.68M/5.68M [00:00<00:00, 5.78MB/s,  554kB/s  ]
tokenizer.json: reconstructing file: 100% 12.8M/12.8M [00:00<00:00, 13.0MB/s, 1.25MB/s  ]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.52MB/s]
model.safetensors.index.json: 100% 50.9k/50.9k [00:00<00:00, 146MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:   9% 163M/1.75G [00:01<00:07, 210MB/s, 12.7MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  20% 342M/1.75G [00:01<00:05, 270MB/s, 29.6MB/s  ]
model.safetensors-00001-of-00001.safeten(…): download

## 6. R0 — the AI with NO lookup ("fail first")
**What:** the model sees only the bank line; it cannot see our customers or invoices.
**Why:** proves the model alone cannot do this job. We expect close to **0%** — that *is* the result.
We run only 40 lines because the outcome is obvious and GPU time is limited.

In [6]:
!python -m pratyaksha.harness.runner --system r0 --split dev --condition clean --limit 40 --cost-mode gpu \
    --opt model=Qwen/Qwen3.5-0.8B --opt batch=8

Loading weights: 100% 473/473 [00:01<00:00, 354.98it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
  [32/40] running accu

## 7. R1 — add lookup (retrieval)
**What:** for each bank line, keyword search finds the **5 most likely customers**, and we give the
model those customers plus their **unpaid invoices on that date** (never future ones).
**Why:** the model can now see the books. We run both clean and messy lines.
**Watch:** *Accuracy*, and *Right party among retrieved* (how often the search found the right customer).
About 3–6 minutes each.

In [7]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r1 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-0.8B --opt batch=8 --opt k=5

Loading weights: 100% 473/473 [00:01<00:00, 343.51it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
  [32/177] running acc

## 8. R2 — add reranking
**What:** search wider (top 10), then a second model (**bge-reranker-v2-m3**, about 2.3 GB) reads the
bank line and each customer name side by side, re-orders them, and we keep only the **best 3**.
**Why:** fewer, better candidates → fewer wrong invoices in the prompt → fewer mistakes, and a shorter
prompt (faster). Does it actually help? That is what we measure.

In [8]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r2 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-0.8B --opt batch=8 --opt keep=3

Loading weights: 100% 473/473 [00:01<00:00, 361.35it/s]
config.json: 100% 795/795 [00:00<00:00, 4.54MB/s]

model.safetensors: downloading bytes:   5% 109M/2.27G [00:01<00:11, 184MB/s, 7.55MB/s  ] 
model.safetensors: downloading bytes:  26% 592M/2.27G [00:03<00:12, 131MB/s, 48.2MB/s  ]
model.safetensors: downloading bytes:  29% 650M/2.27G [00:04<00:14, 111MB/s, 50.2MB/s  ]
model.safetensors: downloading bytes:  35% 794M/2.27G [00:04<00:09, 159MB/s, 59.6MB/s  ]
model.safetensors: downloading bytes:  39% 887M/2.27G [00:05<00:06, 210MB/s, 64.2MB/s  ]
model.safetensors: downloading bytes:  49% 1.11G/2.27G [00:06<00:05, 200MB/s, 78.5MB/s  ]
model.safetensors: downloading bytes:  61% 1.39G/2.27G [00:07<00:04, 181MB/s, 94.0MB/s  ]
model.safetensors: downloading bytes:  67% 1.52G/2.27G [00:11<00:13, 54.5MB/s, 76.9MB/s  ]
model.safetensors: reconstructing file:  92% 2.08G/2.27G [00:13<00:01, 143MB/s, 91.7MB/s  ]
model.safetensors: downloading bytes: 100% 1.52G/1.52G [00:14<00:00, 104MB/s, 76.5MB

## 9. Compare everything in one table
**Why:** this is the result of Step 2 — each rung next to the no-AI baseline.
Columns: accuracy, share handled automatically, error rate on those, speed (p50 seconds),
*party@k* = right customer found, *tok_in* = prompt size, parse errors, business bar.

In [9]:
!python scripts/compare.py --split dev

system                      cond        n     acc    auto  autoErr   p50 s  party@k  tok_in parseErr   bar
R0_Qwen3.5-0.8B             clean      40    0.0%   67.5%   100.0%    0.66        -     320        0  FAIL
R1_Qwen3.5-0.8B             clean     177    0.6%    0.0%     0.0%    1.51    90.6%    1929       77  PASS
R1_Qwen3.5-0.8B             stress    177    1.1%    0.0%     0.0%    1.56    88.3%    1907       73  PASS
R2_Qwen3.5-0.8B             clean     177    0.0%    0.0%     0.0%    1.60    90.6%    1340       34  PASS
R2_Qwen3.5-0.8B             stress    177    0.0%    0.0%     0.0%    1.59    86.0%    1291       27  PASS
rules                       clean     177   80.8%   71.8%     0.0%    0.13        -       -        -  FAIL
rules                       stress    177   59.9%   53.1%     0.0%    0.09        -       -        -  FAIL


## 10. (Optional, if time) Same R2 with the bigger 2B model
**Why:** is the problem the *information* we give the model, or the *size* of the model?
If 2B is much better than 0.8B with the same input, model size matters here.

In [10]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r2 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt batch=8 --opt keep=3
!python scripts/compare.py --split dev

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.64MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 25.8MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 13.3MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 56.2MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 2.39MB/s]
model.safetensors.index.json: 100% 64.5k/64.5k [00:00<00:00, 123MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:   4% 195M/4.55G [00:01<00:19, 223MB/s, 16.2MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:   5% 224M/4.55G [00:01<00:18, 236MB/s, 18.5MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  20% 931M/4.55G [00:04<00:20, 175MB/s, 71.0MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  26% 1.17G/4.55G [00:05<00:15, 215MB/s, 84.1MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  30% 1.35G/4.55G [00:06<00:20, 158MB/s, 91.1MB/s  ]
model.safetensors-00001-of-00001.safeten

In [11]:
import json, glob, collections
for pat in ["*R2_Qwen3.5-2B_dev_clean", "*R1_Qwen3.5-0.8B_dev_clean"]:
    run = sorted(glob.glob(f"runs/{pat}/predictions.jsonl"))[-1]
    rows = [json.loads(l) for l in open(run)]
    errs = collections.Counter(r["pred"]["error"].split(":")[0] for r in rows if r["pred"]["error"])
    print("\n==", run.split("/")[1], dict(errs))
    for r in [r for r in rows if r["pred"]["error"]][:3]:
        print("---", r["pred"]["error"]); print(r["pred"]["raw_output"][:400])


== 20261122-121308_R2_Qwen3.5-2B_dev_clean {'no_json': 23}
--- no_json
{
  "category": "full_payment",
  "party_id": "C051",
  "invoice_ids": [
    "WWI/26-27/00041",
    "WWI/26-27/00077",
    "WWI/26-27/00140",
    "WWI/26-27/00229",
    "WWI/26-27/00267",
    "WWI/26-27/00277",
    "WWI/26-27/00348",
    "WWI
--- no_json
{
  "category": "full_payment",
  "party_id": "C020",
  "invoice_ids": [
    "WWI/26-27/00081",
    "WWI/26-27/00631",
    "WWI/26-27/00642",
    "WWI/26-27/00650",
    "WWI/26-27/00701",
    "WWI/26-27/00719",
    "WWI/26-27/00775",
    "WWI
--- no_json
{
  "category": "full_payment",
  "party_id": "C051",
  "invoice_ids": [
    "WWI/26-27/00041",
    "WWI/26-27/00077",
    "WWI/26-27/00140",
    "WWI/26-27/00229",
    "WWI/26-27/00267",
    "WWI/26-27/00277",
    "WWI/26-27/00348",
    "WWI

== 20261122-115045_R1_Qwen3.5-0.8B_dev_clean {'no_json': 77}
--- no_json
{
  "category": "short_payment",
  "party_id": "C032",
  "invoice_ids": ["WWI/26-27/00031", "WWI/26-2

## 11. Send results back
Download **`runs/index.csv`** from the Drive folder (or screenshot the table from cell 9/10) and send it.
Every run's full detail is in `runs/<run_id>/scorecard.md`.